# Unitree G1 — Flat Terrain Training
Train con robot đi trên mặt phẳng bằng MuJoCo Playground + Brax PPO trên Kaggle Dual T4.

In [1]:
# ============================================================
# Cell 1: Kiểm tra GPU
# ============================================================
import subprocess

subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"])

0, Tesla T4, 15360 MiB
1, Tesla T4, 15360 MiB


CompletedProcess(args=['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv,noheader'], returncode=0)

In [2]:
# ============================================================
# Cell 2: Clone repo & cd vào project
# ============================================================
import os, subprocess
from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/Bipedal-Locomotion")

if not PROJECT_DIR.exists():
    subprocess.check_call([
        "git", "clone",
        "--branch", "M2",
        "https://github.com/TotallyNotMinh/Bipedal-Locomotion.git",
        str(PROJECT_DIR)
    ])
    print("[✓] Cloned repo")
else:
    subprocess.check_call(["git", "-C", str(PROJECT_DIR), "pull"])
    print("[✓] Repo up to date")

os.chdir(PROJECT_DIR)
print("Working dir:", os.getcwd())

Updating 2ae1cd7..f745975
Fast-forward
 configs/larger_mlp.yaml       |  35 ---
 configs/ppo_baseline.yaml     |  71 -----
 configs/ppo_flat.yaml         |  42 +++
 kaggle_remote.ipynb           | 296 +++++++++++++++------
 scripts/eval_m2.py            | 119 ---------
 scripts/record_demo_m2.py     | 171 ------------
 scripts/train_flat.py         | 301 +++++++++++++++++++++
 scripts/train_m2.py           | 149 -----------
 src/models/__init__.py        |  29 ---
 src/models/mlp_policy.py      | 542 --------------------------------------
 src/training/__init__.py      |  27 --
 src/training/config_loader.py |  76 ------
 src/training/env_builder.py   | 259 -------------------
 src/training/ppo_runner.py    | 588 ------------------------------------------
 tests/test_model2_mlp.py      | 106 --------
 train_flat_terrain.ipynb      | 289 ---------------------
 16 files changed, 565 insertions(+), 2535 deletions(-)
 delete mode 100644 configs/larger_mlp.yaml
 delete mode 100644 configs/p

From https://github.com/TotallyNotMinh/Bipedal-Locomotion
   2ae1cd7..f745975  M2         -> origin/M2


In [3]:
# ============================================================
# Cell 3: Cài đặt dependencies
# ============================================================
import sys, subprocess

pkgs = [
    "brax",
    "flax",
    "optax",
    "orbax-checkpoint",
    "ml_collections",
    "pyyaml",
    "imageio",
    "imageio-ffmpeg",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade"] + pkgs)

# MuJoCo Playground (v0.2.0 có G1JoystickFlatTerrain)
try:
    import mujoco_playground
    print("[✓] mujoco_playground already installed")
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade", "--no-deps",
        "git+https://github.com/google-deepmind/mujoco_playground.git@v0.2.0"
    ])
    print("[✓] mujoco_playground installed")

Failed to import warp: No module named 'warp'
Failed to import mujoco_warp: No module named 'warp'
[✓] mujoco_playground already installed


In [4]:
# ============================================================
# Cell 4: Kiểm tra JAX & import
# ============================================================
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.85"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

import jax
devices = jax.devices()
print(f"[✓] JAX {jax.__version__} — {len(devices)} device(s)")
for d in devices:
    print(f"    {d}")

import mujoco_playground
print("[✓] mujoco_playground imported OK")

[✓] JAX 0.11.1 — 2 device(s)
    cuda:0
    cuda:1
[✓] mujoco_playground imported OK


In [5]:
# ============================================================
# Cell 5: TRAIN — robot đi trên mặt phẳng
#
# Thời gian ước tính trên Dual T4:
#   50M steps ~ 40-70 phút
# Checkpoint lưu tại results/flat/g1_flat_<timestamp>/checkpoints/
# ============================================================
import importlib.util
import os
import sys
import yaml

PROJECT_DIR = "/kaggle/working/Bipedal-Locomotion"
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

spec = importlib.util.spec_from_file_location("train_flat", f"{PROJECT_DIR}/scripts/train_flat.py")
train_flat = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train_flat)

with open("configs/ppo_flat.yaml") as f:
    cfg = yaml.safe_load(f)

meta = train_flat.train(cfg, output_dir=cfg["experiment"].get("output_dir", "results/flat"))
print("\n=== DONE ===")
print("Checkpoint:", meta["checkpoint_dir"])
print("Total time:", meta["total_wall_time_s"], "s")

[+] JAX devices (2): ['cuda:0', 'cuda:1']
[+] Output: /kaggle/working/Bipedal-Locomotion/results/flat/g1_flat_20261125_043013
[+] obs_size={'privileged_state': (216,), 'state': (103,)}  action_size=29
[+] Starting PPO training...
  step=         0  elapsed=   274s  sps=         0  eval_reward= -3.508  eval_len=  54.8
  step= 2,785,280  elapsed=   716s  sps=     3,891  eval_reward= -2.048  eval_len=  47.3
  step= 5,570,560  elapsed=   971s  sps=     5,735  eval_reward= -1.807  eval_len=  49.6
  step= 8,355,840  elapsed=  1136s  sps=     7,358  eval_reward= -1.655  eval_len=  55.1
  step=11,141,120  elapsed=  1300s  sps=     8,569  eval_reward= -1.636  eval_len=  58.6
  step=13,926,400  elapsed=  1464s  sps=     9,510  eval_reward= -1.643  eval_len=  60.6
  step=16,711,680  elapsed=  1629s  sps=    10,260  eval_reward= -1.582  eval_len=  63.7
  step=19,496,960  elapsed=  1793s  sps=    10,873  eval_reward= -1.520  eval_len=  67.2
  step=22,282,240  elapsed=  1957s  sps=    11,383  eval_r

In [6]:
# ============================================================
# Cell 6 (tuỳ chọn): Xem metrics sau khi train xong
# ============================================================
import glob, json

run_dirs = sorted(glob.glob("results/flat/g1_flat_*"))
if not run_dirs:
    print("Chưa có kết quả train")
else:
    latest = run_dirs[-1]
    rows = []
    with open(f"{latest}/metrics.jsonl") as f:
        for line in f:
            rows.append(json.loads(line))

    print(f"Run: {latest}")
    print(f"{'step':>12}  {'eval_reward':>12}  {'eval_len':>10}  {'wall_time_s':>12}")
    for r in rows:
        print(
            f"{r.get('step', 0):>12,}  "
            f"{r.get('eval/episode_reward', float('nan')):>12.3f}  "
            f"{r.get('eval/avg_episode_length', float('nan')):>10.1f}  "
            f"{r.get('wall_time_s', 0):>12.0f}"
        )

Run: results/flat/g1_flat_20261125_043013
        step   eval_reward    eval_len   wall_time_s
           0        -3.508        54.8           274
   2,785,280        -2.048        47.3           716
   5,570,560        -1.807        49.6           971
   8,355,840        -1.655        55.1          1136
  11,141,120        -1.636        58.6          1300
  13,926,400        -1.643        60.6          1464
  16,711,680        -1.582        63.7          1629
  19,496,960        -1.520        67.2          1793
  22,282,240        -1.420        74.9          1957
  25,067,520        -1.436        80.3          2122
  27,852,800        -1.311        81.3          2286
  30,638,080        -0.760       114.0          2450
  33,423,360        -0.517       148.7          2615
  36,208,640         0.463       221.4          2779
  38,993,920         3.683       412.1          2944
  41,779,200         5.265       473.9          3108
  44,564,480         6.534       517.5          3272
  47

In [ ]:
# ============================================================
# Cell 7 (tuỳ chọn): Render video demo sau train
# ============================================================
import glob, inspect, os, traceback
from base64 import b64encode
from pathlib import Path

import imageio
import numpy as np
import yaml
from IPython.display import HTML

os.environ["MUJOCO_GL"] = "egl"

import flax.linen as nn
import jax
import mujoco
from brax.training.acme import running_statistics
from brax.training.agents.ppo import checkpoint as ppo_checkpoint
from brax.training.agents.ppo import networks as ppo_networks
from mujoco import mjx
from mujoco_playground import locomotion

# 1. Chọn run mới nhất & resolved checkpoint step
run_dirs = sorted(glob.glob("results/flat/g1_flat_*"))
assert run_dirs, "Chưa có checkpoint!"
latest_run = Path(run_dirs[-1]).resolve()

ckpt_root = latest_run / "checkpoints"
step_dirs = sorted([d for d in ckpt_root.iterdir() if d.is_dir() and d.name.isdigit()], key=lambda d: int(d.name))
ckpt_path = (step_dirs[-1] if step_dirs else ckpt_root).resolve()
print(f"[+] Checkpoint: {ckpt_path}")
print(f"[+] Files in checkpoint: {[p.name for p in ckpt_path.iterdir()]}")

# 2. Load config & environment (ép impl='jax')
cfg_path = latest_run / "resolved_config.yaml"
if not cfg_path.exists():
    cfg_path = Path("configs/ppo_flat.yaml")

if cfg_path.exists():
    with open(cfg_path) as f:
        cfg = yaml.safe_load(f)
else:
    cfg = {
        "env": {"task_name": "G1JoystickFlatTerrain", "impl": "jax"},
        "network": {
            "policy_hidden_layer_sizes": [512, 256, 128],
            "value_hidden_layer_sizes": [512, 256, 128],
            "activation": "swish",
        },
    }

task_name = cfg.get("env", {}).get("task_name", "G1JoystickFlatTerrain")
pg_cfg = locomotion.get_default_config(task_name)
if hasattr(pg_cfg, "impl"):
    pg_cfg.impl = cfg.get("env", {}).get("impl", "jax")
env = locomotion.load(task_name, config=pg_cfg)

# 3. Load policy từ checkpoint
net_cfg = cfg.get("network", {})
policy_hidden = tuple(net_cfg.get("policy_hidden_layer_sizes", [512, 256, 128]))
value_hidden = tuple(net_cfg.get("value_hidden_layer_sizes", [512, 256, 128]))
activation = getattr(nn, net_cfg.get("activation", "swish"), nn.swish)
valid_net_params = set(inspect.signature(ppo_networks.make_ppo_networks).parameters.keys())

def _make_factory(v_key):
    defaults = {
        "policy_hidden_layer_sizes": policy_hidden,
        "value_hidden_layer_sizes": value_hidden,
        "activation": activation,
    }
    if isinstance(env.observation_size, dict):
        if "policy_obs_key" in valid_net_params:
            defaults["policy_obs_key"] = net_cfg.get("policy_obs_key", "state")
        if "value_obs_key" in valid_net_params and v_key in env.observation_size:
            defaults["value_obs_key"] = v_key
    if "distribution_type" in net_cfg and "distribution_type" in valid_net_params:
        defaults["distribution_type"] = net_cfg["distribution_type"]

    def make_networks(*args, **kwargs):
        merged = dict(defaults)
        merged.update({k: v for k, v in kwargs.items() if k in valid_net_params})
        return ppo_networks.make_ppo_networks(*args, **merged)
    return make_networks

policy_fn = None
errors = []

try:
    policy_fn = ppo_checkpoint.load_policy(str(ckpt_path), deterministic=True)
except Exception as e:
    errors.append(f"load_policy(default): {type(e).__name__}: {e}")

if policy_fn is None:
    for v_key in (net_cfg.get("value_obs_key", "privileged_state"), "state"):
        try:
            policy_fn = ppo_checkpoint.load_policy(
                str(ckpt_path), network_factory=_make_factory(v_key), deterministic=True
            )
            break
        except Exception as e:
            errors.append(f"load_policy({v_key}): {type(e).__name__}: {e}")

if policy_fn is None:
    try:
        params = ppo_checkpoint.load(str(ckpt_path))
        if isinstance(params, (tuple, list)) and len(params) >= 2:
            if isinstance(params[0], dict):
                norm_state = running_statistics.RunningStatisticsState(**params[0])
            else:
                norm_state = params[0]
            policy_params = (norm_state, params[1])
        else:
            policy_params = params

        v_key = net_cfg.get("value_obs_key", "privileged_state")
        ppo_net = _make_factory(v_key)(
            env.observation_size,
            int(env.action_size),
            preprocess_observations_fn=running_statistics.normalize,
        )
        policy_fn = ppo_net.make_policy(policy_params, deterministic=True)
    except Exception as e:
        errors.append(f"manual ppo_checkpoint.load: {type(e).__name__}: {e}\n{traceback.format_exc()}")

if policy_fn is None:
    print("Các lỗi gặp phải khi load policy:")
    for err in errors:
        print(" -", err)
    raise RuntimeError(f"Không load được policy từ {ckpt_path}")

print("[✓] Loaded policy thành công!")
jit_policy = jax.jit(policy_fn)
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)

# 4. Rollout 500 bước (10 giây ở 50 Hz)
key = jax.random.PRNGKey(0)
key, rk = jax.random.split(key)
state = jit_reset(rk)
frames_data = [state.data]

print("[+] Simulating 500 steps...")
for t in range(500):
    key, ak = jax.random.split(key)
    action, _ = jit_policy(state.obs, ak)
    state = jit_step(state, action)
    frames_data.append(state.data)
    if bool(state.done):
        print(f"  Episode ended at step {t+1}")
        break

# 5. Render video với camera theo dõi robot
mj_model = getattr(env, "mj_model", getattr(env, "_mj_model", None))
renderer = mujoco.Renderer(mj_model, height=480, width=640)

cam_id = -1
for cname in ("track", "side", "back"):
    try:
        cid = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_CAMERA, cname)
        if cid >= 0:
            cam_id = cid
            break
    except Exception:
        pass
if cam_id < 0 and mj_model.ncam > 0:
    cam_id = 0

mj_data = mujoco.MjData(mj_model)
frames = []
for d in frames_data:
    try:
        mjx.get_data_into(result=mj_data, model=mj_model, data=d)
    except Exception:
        mj_data.qpos[:] = np.asarray(d.qpos)
        mj_data.qvel[:] = np.asarray(d.qvel)
        mujoco.mj_forward(mj_model, mj_data)
    if cam_id >= 0:
        renderer.update_scene(mj_data, camera=cam_id)
    else:
        renderer.update_scene(mj_data)
    frames.append(renderer.render())

video_path = "/kaggle/working/g1_flat_demo.mp4"
imageio.mimsave(video_path, frames, fps=30, macro_block_size=None)
print(f"[✓] Video: {video_path} ({len(frames)} frames)")

mp4 = open(video_path, "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
display(HTML(f'<video width="640" height="480" controls autoplay loop><source src="{data_url}" type="video/mp4"></video>'))